# F2_03 · Transformación del subconjunto de La Araucanía

**Objetivo:** A partir del subconjunto de egresos hospitalarios de residentes de La Araucanía (generado en `F2_01_obtencion.ipynb`), filtrar los casos de colelitiasis (K80.0 a K80.8) y construir las variables que se usarán en el análisis: provincia de residencia, los grupos de riesgo definidos por el equipo y la marca de colecistectomía.

**Entrada:** `data/processed/egresos_araucania_2022.csv`

**Salida:** `data/processed/egresos_araucania_k80_transformado.csv`

**Qué hace este notebook:**
1. Filtra los egresos cuyo diagnóstico principal (`DIAG1`) corresponde a colelitiasis, códigos CIE-10 K80.0 a K80.8.
2. Crea la variable `PROVINCIA_RESIDENCIA` a partir de la comuna de residencia (comunas 091xx = Cautín, 092xx = Malleco).
3. Crea tres variables de grupo de riesgo, por separado: sexo mujer, pertenencia a alguna etnia, y grupo etario 40 a 49 años.
4. Crea la variable `FLAG_COLECISTECTOMIA`, que identifica si la intervención quirúrgica principal fue una colecistectomía.
5. Guarda el resultado y lo verifica volviendo a leerlo.

**Qué NO hace este notebook:** No realiza la validación final del dataset; eso se hace en `F2_04_validacion.ipynb`.

In [1]:
import pandas as pd
from pathlib import Path

# El notebook está en F2, así que la raíz del proyecto es la carpeta de arriba
RAIZ = Path.cwd()
if RAIZ.name.startswith("F"):
    RAIZ = RAIZ.parent

ENTRADA = RAIZ / "data" / "processed" / "egresos_araucania_2022.csv"
SALIDA = RAIZ / "data" / "processed" / "egresos_araucania_k80_transformado.csv"

print("¿Existe la entrada?", ENTRADA.exists())
print(ENTRADA)

¿Existe la entrada? True
/home/claude/paulabecerrav/proyecto-grupo2-mcdi500/data/processed/egresos_araucania_2022.csv


In [2]:
araucania = pd.read_csv(ENTRADA, dtype=str)

print("Filas y columnas:", araucania.shape)
araucania.head()

Filas y columnas: (84618, 18)


,PERTENENCIA_ESTABLECIMIENTO_SALUD,SEXO,GRUPO_EDAD,ETNIA,GLOSA_PAIS_ORIGEN,COMUNA_RESIDENCIA,GLOSA_COMUNA_RESIDENCIA,REGION_RESIDENCIA,GLOSA_REGION_RESIDENCIA,PREVISION,GLOSA_PREVISION,ANO_EGRESO,DIAG1,DIAG2,DIAS_ESTADA,CONDICION_EGRESO,GLOSA_INTERV_Q_PPAL,GLOSA_PROCED_PPAL
0,No Pertenecientes al Sistema Nacional de Servi...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,09101,Temuco,09,De La Araucanía,1,FONASA,2022,J353,NaN,1,1,"Amigdalectomía c/s adenoidectomía, uni o bilat...",NaN
1,No Pertenecientes al Sistema Nacional de Servi...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,09101,Temuco,09,De La Araucanía,1,FONASA,2022,J353,NaN,1,1,"Amigdalectomía c/s adenoidectomía, uni o bilat...",NaN
2,No Pertenecientes al Sistema Nacional de Servi...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,09202,Collipulli,09,De La Araucanía,1,FONASA,2022,Q531,NaN,1,1,Descenso testículo inguinal c/s hernioplastía,NaN
3,No Pertenecientes al Sistema Nacional de Servi...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,09101,Temuco,09,De La Araucanía,1,FONASA,2022,N47X,NaN,1,1,"Circuncisión (incluye sección de frenillo, y/o...",NaN
4,No Pertenecientes al Sistema Nacional de Servi...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,09101,Temuco,09,De La Araucanía,1,FONASA,2022,H741,NaN,1,1,Mucositis timpánica o mixiosis uni o bilateral...,NaN


## Paso 1 · Filtro de colelitiasis (K80.0 a K80.8)

El diagnóstico principal está en la columna `DIAG1`, con el código CIE-10 sin punto (por ejemplo `K802` en vez de `K80.2`). Se filtran todos los egresos cuyo `DIAG1` empieza con `K80`, lo que incluye los subcódigos K80.0 a K80.8.

In [3]:
es_colelitiasis = araucania["DIAG1"].str.startswith("K80", na=False)
k80 = araucania[es_colelitiasis].copy()

print("Egresos por colelitiasis (K80.0-K80.8):", len(k80))
print()
print("Distribución por subcódigo:")
print(k80["DIAG1"].value_counts().sort_index())

Egresos por colelitiasis (K80.0-K80.8): 3789

Distribución por subcódigo:
DIAG1
K800     383
K801     466
K802    1885
K803      60
K804     115
K805     427
K808     453
Name: count, dtype: int64


## Paso 2 · Variable provincia

La región de La Araucanía tiene dos provincias. El código de comuna (`COMUNA_RESIDENCIA`) indica la provincia en sus primeros tres dígitos:

- `091xx` → provincia de Cautín
- `092xx` → provincia de Malleco

Con eso se construye la variable `PROVINCIA_RESIDENCIA`.

In [4]:
provincia_por_codigo = {
    "091": "Cautín",
    "092": "Malleco",
}

k80["PROVINCIA_RESIDENCIA"] = k80["COMUNA_RESIDENCIA"].str[:3].map(provincia_por_codigo)

print(k80["PROVINCIA_RESIDENCIA"].value_counts(dropna=False))

sin_provincia = k80["PROVINCIA_RESIDENCIA"].isna().sum()
print()
print("Egresos sin provincia asignada:", sin_provincia)

PROVINCIA_RESIDENCIA
Cautín     3017
Malleco     772
Name: count, dtype: int64

Egresos sin provincia asignada: 0


## Paso 3 · Grupos de riesgo

El equipo definió tres grupos de riesgo para colelitiasis, que se registran como tres variables independientes (no combinadas entre sí), para poder analizarlas por separado:

- `GRUPO_RIESGO_MUJER`: sexo mujer.
- `GRUPO_RIESGO_ETNIA`: se identifica con alguna etnia.
- `GRUPO_RIESGO_40_49`: grupo etario de 40 a 49 años.

In [5]:
k80["GRUPO_RIESGO_MUJER"] = k80["SEXO"] == "MUJER"
k80["GRUPO_RIESGO_ETNIA"] = k80["ETNIA"] == "Si se identifica con alguna etnia"
k80["GRUPO_RIESGO_40_49"] = k80["GRUPO_EDAD"] == "40 a 49"

print("Mujeres:", k80["GRUPO_RIESGO_MUJER"].sum())
print("Se identifica con alguna etnia:", k80["GRUPO_RIESGO_ETNIA"].sum())
print("Grupo etario 40 a 49:", k80["GRUPO_RIESGO_40_49"].sum())

Mujeres: 2764
Se identifica con alguna etnia: 753
Grupo etario 40 a 49: 648


## Paso 4 · Marca de colecistectomía

Se busca la palabra "colecistectom" (sin acento, para no depender de mayúsculas ni de cómo esté escrita la tilde) dentro de `GLOSA_INTERV_Q_PPAL`, la glosa de la intervención quirúrgica principal. Esto captura todas las variantes de colecistectomía registradas (por videolaparoscopía, con o sin colangiografía, etc.), sin incluir otras intervenciones de la vesícula que no son colecistectomía, como la colecistostomía.

In [6]:
k80["FLAG_COLECISTECTOMIA"] = k80["GLOSA_INTERV_Q_PPAL"].str.contains(
    "colecistectom", case=False, na=False
)

print("Egresos con colecistectomía:", k80["FLAG_COLECISTECTOMIA"].sum())
print("Egresos sin colecistectomía:", (~k80["FLAG_COLECISTECTOMIA"]).sum())
print()
print("Intervenciones más frecuentes marcadas como colecistectomía:")
print(k80.loc[k80["FLAG_COLECISTECTOMIA"], "GLOSA_INTERV_Q_PPAL"].value_counts())

Egresos con colecistectomía: 3056
Egresos sin colecistectomía: 733

Intervenciones más frecuentes marcadas como colecistectomía:
GLOSA_INTERV_Q_PPAL
Colecistectomía por videolaparoscopía, proc. completo                                                       2988
Colecistectomía c/s colangiografía operatoria                                                                 53
Colecistectomía y coledocostomía (sonda T y colangiografía postoperatoria) c/s colangiografía operatoria      15
Name: count, dtype: int64


## Guardar y verificar

In [7]:
k80.to_csv(SALIDA, index=False, encoding="utf-8")

print("Guardado en:", SALIDA)
print("Filas y columnas guardadas:", k80.shape)

Guardado en: /home/claude/paulabecerrav/proyecto-grupo2-mcdi500/data/processed/egresos_araucania_k80_transformado.csv
Filas y columnas guardadas: (3789, 23)


In [8]:
verificacion = pd.read_csv(SALIDA, dtype=str)

# Las columnas booleanas se leen como texto ("True"/"False") al volver a leer el CSV,
# por eso se comparan como string para la verificación.
k80_como_texto = k80.astype(str).reset_index(drop=True)
verificacion_como_texto = verificacion.astype(str).reset_index(drop=True)

print("Filas y columnas del archivo guardado:", verificacion.shape)
print("¿Es idéntico a lo que transformamos?", k80_como_texto.equals(verificacion_como_texto))

Filas y columnas del archivo guardado: (3789, 23)
¿Es idéntico a lo que transformamos? True


## Resultado

- Egresos por colelitiasis (K80.0-K80.8) en La Araucanía: 3789 (de 84618 egresos totales de la región).
- Por provincia: 3017 en Cautín, 772 en Malleco. Todos los egresos quedaron con provincia asignada.
- Grupos de riesgo: 2764 mujeres, 753 con alguna etnia, 648 en el grupo etario 40 a 49 años.
- Colecistectomía: 3056 egresos con colecistectomía registrada, 733 sin ella.
- Subconjunto transformado guardado en `data/processed/egresos_araucania_k80_transformado.csv` y verificado al volver a leerlo.

**Para reproducir:** ejecutar primero `F2_01_obtencion.ipynb` y luego este notebook de principio a fin.

## Paso 5 · Funciones reutilizables (`src/transformacion.py`)

El filtro y las variables de los pasos anteriores se reescribieron como funciones con parámetros, en `src/transformacion.py`, para que sean reutilizables (y sirvan de base a las clases de la Fase 3). Cada función recibe un DataFrame, no lo modifica (trabaja sobre una copia) y devuelve un DataFrame nuevo, así el flujo de datos es verificable paso a paso: `filtrar_colelitiasis`, `asignar_provincia`, `marcar_grupos_riesgo`, `marcar_colecistectomia`, y `transformar`, que encadena las cuatro.

In [9]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
if RAIZ.name.startswith("F"):
    RAIZ = RAIZ.parent

sys.path.insert(0, str(RAIZ))
import src.transformacion as tr

### Verificación: ¿da el mismo resultado que el filtro manual de arriba?

In [10]:
resultado_funciones = tr.transformar(araucania)

k80_texto = k80.astype(str).reset_index(drop=True)
resultado_texto = resultado_funciones.astype(str).reset_index(drop=True)

print("Filas y columnas con las funciones:", resultado_funciones.shape)
print("¿Mismas columnas, mismo orden?", list(resultado_texto.columns) == list(k80_texto.columns))
print("¿Es idéntico al resultado del filtro manual?", resultado_texto.equals(k80_texto))

Filas y columnas con las funciones: (3789, 23)
¿Mismas columnas, mismo orden? True
¿Es idéntico al resultado del filtro manual? True


### Pruebas: caso normal, caso límite y excepción

Caso normal: ya se verificó arriba, con los 3.789 egresos reales.

In [11]:
# Caso límite: un código de comuna que no es 091 ni 092, y una glosa de
# intervención vacía o nula. No debe reventar, y el resultado debe ser
# coherente: provincia nula para el código que no corresponde a ninguna de
# las dos provincias, y colecistectomía falsa cuando la glosa está vacía.
limite_provincia = pd.DataFrame({"COMUNA_RESIDENCIA": ["09101", "08101", "*"]})
r_provincia = tr.asignar_provincia(limite_provincia)
print("Provincias asignadas:", r_provincia["PROVINCIA_RESIDENCIA"].tolist())
assert r_provincia["PROVINCIA_RESIDENCIA"].isna().sum() == 2, "Los códigos fuera de 091/092 deben quedar sin provincia"

limite_glosa = pd.DataFrame({"GLOSA_INTERV_Q_PPAL": ["", None, "COLECISTECTOMIA LAPAROSCOPICA"]})
r_glosa = tr.marcar_colecistectomia(limite_glosa)
print("Colecistectomía marcada:", r_glosa["FLAG_COLECISTECTOMIA"].tolist())
assert r_glosa["FLAG_COLECISTECTOMIA"].tolist() == [False, False, True], "Vacío o nulo debe quedar en False"

Provincias asignadas: ['Cautín', nan, nan]
Colecistectomía marcada: [False, False, True]


In [12]:
# Excepción: si al conjunto le falta una columna que la función necesita,
# debe fallar con un error claro (KeyError) y no con un error confuso de pandas.
try:
    tr.marcar_colecistectomia(pd.DataFrame({"otra_columna": [1, 2]}))
    print("No debería llegar aquí")
except KeyError as e:
    print("Error esperado y capturado:", e)

Error esperado y capturado: 'marcar_colecistectomia: falta la columna GLOSA_INTERV_Q_PPAL.'


### Resultado

- Las funciones de `src/transformacion.py` producen exactamente el mismo resultado que el filtro manual de este notebook (3.789 filas, 23 columnas, columnas idénticas).
- El caso límite (código de comuna fuera de 091/092, glosa de intervención vacía o nula) se maneja sin errores y con el resultado esperado.
- La excepción por columna faltante se captura con un mensaje claro en vez de un error genérico de pandas.
- Esto deja la lógica de la transformación lista para reutilizarse en la Fase 3, dentro de las clases del pipeline.